# **ESMC를 사용한 임베딩 추출**

2024315315 권효재

---

기존의 단백질 모델이

*   sequence를 다루는 모델(PLM)
*   structure을 다루는 모델
*   function을 다루는 모델

과 같이 별도로 존재하였던 것과 달리, ESM3는 한 모델 안에서 전부 가능하게 하는, 즉 input과 output으로 여러 Properties를 주고받는 Multiomodal 모델이다.

기본적으로 BERT와 유사하게 일부 서열을 mask하고 예측하게 하는 방식으로 학습/생성한다.


**목차**
1. 서열 > ESMC 모델 인풋 인코딩
2. 각 위치에서 어떤 아미노산이 나올 지 확률 분포 예측
3. 예측한 확률 분포를 기반으로 임베딩 추출


---



# esm.models.esmc

*   ESMC: 학습을 위해 사용되는 모델이다.

# esm.sdk.api

properties를 discrete tokens로 용이하게 다루기 위해 사용하는 sdk다.

*   ESMProtein: 5개의 traks를 담고 있는 object class다.




In [ ]:
#ESM3/ESMC는 파이썬 내 esm 패키지로 설치할 수 있다.
!pip install esm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.5 MB/s

In [ ]:
#from huggingface_hub import login
from esm.models.esmc import ESMC
#from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, LogitsConfig, LogitsOutput, ESMCInferenceClient, ESM3InferenceClient, ESMProteinError
#login()
#from pathlib import Path
#print(LogitsConfig)
#help(ESMCInferenceClient)

In [ ]:
import torch
#device = "cuda" if torch.cuda.is_available() else "cpu"
clientC = ESMC.from_pretrained()
#client3 = ESM3.from_pretrained()
#model = model.to(device).eval()
#print(ESMC.from_pretrained())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

data/weights/esmc_600m_2024_12_v0.pth:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

In [ ]:
testprotein1 = "MSTDSDAETVDLADGVDHQVAMVMDLNKCIGCQTCTVACKSLWTEGGGRDYMYWNNVETKPGKGYPRNWEESGGGWKSSEHKERKPGQIPDKEDYGDAWEFNHEEIMYNGSDRPLRPDSDPEWGPNWDEDQGTGEYPNSYYFYLPRICNHCTHPSCVEACPRKAIYKREEDGIVLIDQERCRGYRYCVEGCPYKKVYYNATQKTSEKCIFCYPRIEGEGPDGKTFAPACAEDCPPQLRLVGFLDDEQGPIHKLVEEYEVALPLHPEYQTQPNVYYIPPFAPPQHSEDGESVDVDRIPRNYLEELFGERVHDALDTIEREREKVNRGGGSELLDMLTDTNPARKFRLEVFDDD"
testprotein2 = ["MSTDSDAETVDLADGVDHQVAMVMDLNKCIGCQTCTVACKSLWTEGGGRDYMYWNNVETKPGKGYPRNWEESGGGWKSSEHKERKPGQIPDKEDYGDAWEFNHEEIMYNGSDRPLRPDSDPEWGPNWDEDQGTGEYPNSYYFYLPRICNHCTHPSCVEACPRKAIYKREEDGIVLIDQERCRGYRYCVEGCPYKKVYYNATQKTSEKCIFCYPRIEGEGPDGKTFAPACAEDCPPQLRLVGFLDDEQGPIHKLVEEYEVALPLHPEYQTQPNVYYIPPFAPPQHSEDGESVDVDRIPRNYLEELFGERVHDALDTIEREREKVNRGGGSELLDMLTDTNPARKFRLEVFDDD", "MSRNDASQLDDGETTAESPPDDQANDAPEVGDPPGDPVDADSGVSRRTFLEGIGVASLLGIGTSAASDDSLFQMGGLKPVDDPIGNYPYRDWEDLYREKWDWDSVSRSTHSVNCTGSCSWNVYVKNGQVWREEQSGDYPRFDESLPDPNPRGCQKGACYTDYVNADQRIKHPLKRVGERGEGKWKRISWDEALTEIAEHVVDEVEAGRYDAISGFTPIPAMSPVSFASGSRLVNLLGGVSHSFYDWYSDLPPGQPITWGTQTDNAESADWYNADYIIAWGSNINVTRIPDAKYFLESGYNGTKRVGVFTDYSQTAIHTDEWLSPDSGTDTALALGMAQTIVDEGLYDEAHLKEQTDMPLLVRQDTGKFLRASDVPSVNTDADRPEWMLLMLDSNGRIREAPGSLGERDGQKDYSKSIELDFDPQLDGETTVQTQSGRVQVRTVWAELRDELANWDPEMVHEETTVGKETYQRIAREFAEADKAKIIQGKGVNDWYHNDLGNRALQLLVTLTGNLGEQGTGLDHYVGQEKIWTFHGWKTLSFPTGKVRGVPTTLWTYYHAGILDNTDPDTAAKIRESIDKGWMPVYPEERDNGSRPDPTTMFVWRGNYFNQAKGNVAVEEQLWPKLDLVVDINFRMDSTAMYSDIVLPTASHYEKHDLSMTDMHTYVHPFTPAVEPLGESKTDWQIFRELAQKIQEVATERGVEPISDRKFDREIDLQSVYDDYVRDWETGEEGALAEDRAACEYILEHSEESNPADSDEQITFADTVEQPQRLLEAGDHWTSDIEDGEAYAPWKDFVQDKNPWPTVTGRQQYYIDHDWFLELGEELPTHKEGPENTGGDYPMEYNTPHGRWAIHSTWRDSEKLLRLQRGEPLLYLHPEDAEERGIEDGDSVEVFNDLAEVELQAKIYPSSQRGTARMYFAWERFQFDSDSNFNSLVPMYMKPTQLVQYPEDSGEHLHFFPNYWGPTGVNSDVRVDVRKAGGGDE"]

In [ ]:
def ESMC_600M_202412(device: torch.device | str = "cpu", use_flash_attn: bool = True):
    with torch.device(device):
        model = ESMC(
            d_model=1152,
            n_heads=18,
            n_layers=36,
            tokenizer=get_esmc_model_tokenizers(),
            use_flash_attn=use_flash_attn,
        ).eval()
    state_dict = torch.load(
        data_root("esmc-600") / "data/weights/esmc_600m_2024_12_v0.pth",
        map_location=device,
    )
    model.load_state_dict(state_dict)

    return model

class ESMC(nn.Module, ESMCInferenceClient):
    """
    ESMC model implementation.

    Args:
        d_model (int): The dimensionality of the input and output feature vectors.
        n_heads (int): The number of attention heads in the transformer layers.
        n_layers (int): The number of transformer layers.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_layers: int,
        tokenizer: EsmSequenceTokenizer,
        use_flash_attn: bool = True,
    ):
        super().__init__()
        self.embed = nn.Embedding(64, d_model)

        self._use_flash_attn = is_flash_attn_available and use_flash_attn
        self.transformer = TransformerStack(
            d_model,
            n_heads,
            None,
            n_layers,
            n_layers_geom=0,
            use_flash_attn=self._use_flash_attn,
        )

        self.sequence_head = RegressionHead(d_model, 64)
        self.tokenizer = tokenizer

    def encode(self, input: ESMProtein) -> ESMProteinTensor:
        input = attr.evolve(input)  # Make a copy
        sequence_tokens = None

        if input.sequence is not None:
            sequence_tokens = self._tokenize([input.sequence])[0]
        return ESMProteinTensor(
            sequence=sequence_tokens,
            potential_sequence_of_concern=input.potential_sequence_of_concern,
        ).to(next(self.parameters()).device)

    def _tokenize(self, sequence: list[str]) -> torch.Tensor:
        pad = self.tokenizer.pad_token_id
        assert pad is not None
        return stack_variable_length_tensors(
            [
                encoding.tokenize_sequence(x, self.tokenizer, add_special_tokens=True)
                for x in sequence
            ],
            constant_value=pad,
        ).to(next(self.parameters()).device)

    def tokenize_sequence(
    sequence: str,
    sequence_tokenizer: EsmSequenceTokenizer,
    add_special_tokens: bool = True,
) -> torch.Tensor:
    sequence = sequence.replace(C.MASK_STR_SHORT, sequence_tokenizer.mask_token)
    sequence_tokens = sequence_tokenizer.encode(
        sequence, add_special_tokens=add_special_tokens
    )
    sequence_tokens = torch.tensor(sequence_tokens, dtype=torch.int64)
    return sequence_tokens

In [ ]:
from esm.models.esmc import EsmSequenceTokenizer


testprotein = ESMProtein(sequence=testprotein1)
testproteinTensor = clientC.encode(testprotein)
output = clientC.logits(testproteinTensor, LogitsConfig(return_embeddings=True))
embed = output.embeddings
print(embed)



#tokenizer = EsmSequenceTokenizer()
#print(tokenizer.convert_ids_to_tokens(testproteinTensor.sequence))
#print("len(seq) =", len(testprotein.sequence), "embedding length =", embed.shape[1])

torch.Size([1, 352, 1152])


In [ ]:
#배치 사이즈를 늘려서 해보자..
#먼저 임베딩 추출을 진행하는 함수를 선언하자


def embed_sequence(client: ESM3InferenceClient, sequence: str) -> LogitsOutput:
  protein = ESMProtein(sequence=sequence)
  proteinTensor = client.encode(protein)
  output = client.logits(proteinTensor, LogitsConfig(return_embeddings=True))
  embed = output.embeddings
  return embed

from esm.sdk import batch_executor
with batch_executor() as executor:
  outputs = executor.execute_batch(user_func = embed_sequence,
                                   client = clientC,
                                   sequence = testprotein2)

Processing  100%|████████████████████████| 2/2 [Elapsed: 00:00 | Remaining: 00:00] , Success=2 Fail=0 Retry=0


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 354 but got size 986 for tensor number 1 in the list.

# tqdm

남은 시간과 처리 속도를 보여주는 함수

반복문의 range를 tqdm으로 감싸주면 됨!

In [ ]:
from tqdm import tqdm

In [ ]:
#추출한 임베딩 저장
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# os

파이썬으로 운영체제 시스템 기능 제어할 때 사용하는 라이브러리


.getcwd() 현재 위치 반환

.chdir('경로') ~cd

.mkdir('폴더명') 폴더만들기

.rename('이전이름', '바꿀이름')

.remove('삭제할파일명')

.makedirs('경로', exist_ok=True) 경로를 전부 구현, 이미 있어도 괜찮음

.listdif('.') '.'은 현재 폴더, 폴더 내 파일을 리스트 형태로 반환

.path.join('경로1', '경로2') 경로 잘 합쳐줌 > 파일경로, 파일명 합쳐서 반환시키면 good

.path.exists('파일명') 파일 유뮤 bool 반환


In [ ]:
import os
SAVE_PATH = '/content/drive/MyDrive/ESMC_embedding'
os.makedirs(SAVE_PATH, exist_ok=True)

# h5py

.h5 .hdf5 파일로 저장됨

scientific data 담는 format

대규모 / 압축 / 계층적 구조 유지

(RAM에 올리지 않아도 디스크 슬라이싱으로 읽어올 수 있)

넘파이 인터페이스와 거의 유사하기 때문에 넘파이 배열을 저장하고 사용하는 데 사용할 수 있

파일 자체가 하나의 디렉터리 시스템처럼 작동함 (~리눅스 루트 디렉터리)



*   Group       : 폴더
*   Dataset     : 폴더 속 파일 (다차원 배열)
*   Attributes  : 데이터 설명 / 메타데이터


h5py.File('파일', 'mode')



*   'r'   : 읽기 전용 (기본값)
*   'w'   : 초기화 + 새로 생성
*   'a'   : 내용 유지 + 데이터 추가
*   'r+'  : 내용 유지 + 수정 가능

'r', 'r+' : 파일이 없을 시 에러 반환



In [ ]:
!pip install h5py
import h5py
#help(h5py.File)

Help on class File in module h5py._hl.files:

class File(h5py._hl.group.Group)
 |  File(name, mode='r', driver=None, libver=None, userblock_size=None, swmr=False, rdcc_nslots=None, rdcc_nbytes=None, rdcc_w0=None, track_order=None, fs_strategy=None, fs_persist=False, fs_threshold=1, fs_page_size=None, page_buf_size=None, min_meta_keep=0, min_raw_keep=0, locking=None, alignment_threshold=1, alignment_interval=1, meta_block_size=None, *, track_times=False, **kwds)
 |
 |  Represents an HDF5 file.
 |
 |  Method resolution order:
 |      File
 |      h5py._hl.group.Group
 |      h5py._hl.base.HLObject
 |      h5py._hl.base.CommonStateObject
 |      h5py._hl.base.MutableMappingHDF5
 |      h5py._hl.base.MappingHDF5
 |      collections.abc.MutableMapping
 |      collections.abc.Mapping
 |      collections.abc.Collection
 |      collections.abc.Sized
 |      collections.abc.Iterable
 |      collections.abc.Container
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __enter__(self)
 |


In [ ]:
#저장할 파일명
testfilename = 'test.h5'
#경로+파일명 합쳐서
testh5py = os.path.join(SAVE_PATH, testfilename)

#with로 열면 알아서 open > close
with h5py.File(testh5py, 'a') as f:
  #파일 속 그룹을 생성
  #create_group은 이미 있으면 오류 반환
  testgroup = f.require_group('testgroup1')
  #만든 그룹 내에 데이터셋 생성
  #이미 데이터셋이 있으면 삭제
  if 'testdataset1' in testgroup:
    del testgroup['testdataset1']
  testdataset = testgroup.create_dataset('testdataset1', data = embed.cpu(), compression="gzip")

with h5py.File(testh5py, 'r') as f:
  print(f['testgroup1']['testdataset1'][:])

[[ 0.04798162 -0.02044947 -0.01536392 ... -0.03842252 -0.01144048
   0.00991298]
 [ 0.04568135 -0.00845654 -0.02692959 ... -0.03221452 -0.01734971
   0.01244055]
 [-0.00445518 -0.05785549 -0.03266174 ... -0.05671361  0.01064723
  -0.01733937]
 ...
 [ 0.01436075 -0.04156201 -0.00886083 ... -0.02553995  0.04094667
  -0.029923  ]
 [ 0.00276651 -0.03619592 -0.03828207 ... -0.05944339  0.05693217
  -0.03484154]
 [ 0.02008948 -0.08265961 -0.00710448 ... -0.04350155  0.00208534
  -0.0785713 ]]


In [ ]:
import pandas as pd
df = pd.read_csv('https://github.com/Rainbowbarkbark/Mprotein_hydrophobic/blob/main/Swissprot_Membrane_Train_Validation_dataset.csv?raw=true')
print(df.head(10))

   Unnamed: 0     ACC   Kingdom  Partition  Peripheral  Transmembrane  \
0           0  I3R9M8   Archaea          0           1              0   
1           1  I3R9M9   Archaea          1           1              0   
2           2  Q7ZAG8   Archaea          2           1              0   
3           3  Q8PZ67   Archaea          0           1              0   
4           4  Q9YGA6   Archaea          0           1              0   
5           5  A1RVM8   Archaea          4           1              0   
6           6  D4GYW6   Archaea          1           1              0   
7           7  Q4J923   Archaea          2           1              0   
8           8  P84622   Archaea          0           1              0   
9           9  A0QWG5  Bacteria          1           1              0   

   LipidAnchor  Soluble                                           Sequence  
0            0        0  MSTDSDAETVDLADGVDHQVAMVMDLNKCIGCQTCTVACKSLWTEG...  
1            0        0  MSRNDASQLDDGETTAE

In [ ]:
!pip install h5py
import h5py
import numpy as np

label_cols = ['Partition', 'Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
save_path = "protein_data_with_labels.h5"

with h5py.File(save_path, 'w') as f:

    with torch.inference_mode(): #그래디언트 계산 끄고 역전파X, 가중치 변환X, 기록X
        # DataFrame의 각 행(row)을 하나씩 가져옵니다.
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):

            try:
                acc = row['ACC']       # 단백질 ID
                seq = row['Sequence']  # 서열

                # 라벨 정보 추출 (예: [0, 1, 0, 0, 0])
                # values로 가져와서 numpy array로 변환
                labels = row[label_cols].values.astype(int)

                # --- 임베딩 추출 과정 ---
                #if len(seq) > 2000: continue # 너무 긴 것 스킵(선택)

                protein = ESMProtein(sequence=seq)
                proteinTensor = model.encode(protein)
                #proteinTensor = proteinTensor.to(device) # 필요시 주석 해제

                output = model.logits(proteinTensor, LogitsConfig(return_embeddings=True))

                # [선택 1] 전체 시퀀스 저장 (용량 큼) -> h5py 필수
                embed = output.embeddings[:, 1:-1, :].squeeze(0).cpu()

                # [선택 2] 평균(Mean)만 저장할 거라면 아래 주석 해제 (용량 작음)
                # embed = output.embeddings[:, 1:-1, :].mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)

                # --- HDF5 저장 ---
                # 1. 데이터셋 생성 (이름: ACC, 내용: 임베딩)
                dset = f.create_dataset(str(acc), data=embed, compression="gzip")

                # 2. ***핵심*** 라벨 정보를 속성(Attributes)으로 심어주기
                dset.attrs['targets'] = labels

            except Exception as e:
                print(f"Error processing {index}: {e}")
                continue

print(f"\n저장 완료: {save_path}")
print("이제 이 파일 하나만 있으면 임베딩(X)과 라벨(Y)을 바로 꺼낼 수 있습니다.")


Processing:   0%|          | 50/28026 [00:19<3:03:40,  2.54it/s]


KeyboardInterrupt: 

In [ ]:
'''
# 저장된 파일 확인하기
with h5py.File("protein_data_with_labels.h5", 'r') as f:
    # 첫 번째 단백질 ID 아무거나 가져와보기
    sample_acc = list(f.keys())[0]

    data = f[sample_acc] # 데이터셋 접근

    print(f"ID: {sample_acc}")
    print(f"임베딩 모양(X): {data[:].shape}") # (Sequence_len, 1024)
    print(f"라벨 정보(Y): {data.attrs['targets']}") # [0 1 0 0 0]
    '''

'\n# 저장된 파일 확인하기\nwith h5py.File("protein_data_with_labels.h5", \'r\') as f:\n    # 첫 번째 단백질 ID 아무거나 가져와보기\n    sample_acc = list(f.keys())[0]\n    \n    data = f[sample_acc] # 데이터셋 접근\n    \n    print(f"ID: {sample_acc}")\n    print(f"임베딩 모양(X): {data[:].shape}") # (Sequence_len, 1024)\n    print(f"라벨 정보(Y): {data.attrs[\'targets\']}") # [0 1 0 0 0]\n    '

In [ ]:
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer

def get_default_sequence_tokens(
    sequence_length: int, sequence_tokenizer: EsmSequenceTokenizer
) -> torch.Tensor:
    assert sequence_tokenizer.mask_token_id is not None
    assert sequence_tokenizer.bos_token_id is not None
    assert sequence_tokenizer.eos_token_id is not None

    sequence_tokens = torch.full(
        (sequence_length + 2,), sequence_tokenizer.mask_token_id
    )
    sequence_tokens[0] = sequence_tokenizer.bos_token_id
    sequence_tokens[-1] = sequence_tokenizer.eos_token_id

    return sequence_tokens

def get_default_sequence(sequence_length: int) -> str:
    return C.MASK_STR_SHORT * sequence_length
MASK_STR_SHORT = "_"

def tokenize_sequence(
    sequence: str,
    sequence_tokenizer: EsmSequenceTokenizer,
    add_special_tokens: bool = True,
) -> torch.Tensor:
    sequence = sequence.replace(C.MASK_STR_SHORT, sequence_tokenizer.mask_token)
    sequence_tokens = sequence_tokenizer.encode(
        sequence, add_special_tokens=add_special_tokens
    )
    sequence_tokens = torch.tensor(sequence_tokens, dtype=torch.int64)
    return sequence_tokens